# SAP AI Core: Prompt Optimization Demo (Minimal)

End-to-end workflow: Live orchestration → Evaluation → Optimization

## 1. Setup

In [1]:
%pip install httpx boto3 python-dotenv generative-ai-hub-sdk --quiet


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import json, mimetypes, os, re, time
from pathlib import Path
from urllib.parse import quote
import httpx, boto3
from dotenv import load_dotenv
from gen_ai_hub.orchestration_v2.models.config import ModuleConfig, OrchestrationConfig
from gen_ai_hub.orchestration_v2.models.llm_model_details import LLMModelDetails
from gen_ai_hub.orchestration_v2.models.message import SystemMessage, UserMessage
from gen_ai_hub.orchestration_v2.models.template import PromptTemplatingModuleConfig, Template
from gen_ai_hub.orchestration_v2.service import OrchestrationService

load_dotenv(override=True)

# Configuration
USE_CASE_NAME = "delivery-delay-reply-coach"
RUN_ID = time.strftime("%Y%m%d-%H%M%S")
LOCAL_WORKDIR = Path("generated") / RUN_ID
LOCAL_WORKDIR.mkdir(parents=True, exist_ok=True)

# Environment
AICORE_AUTH_URL = os.environ["AICORE_AUTH_URL"]
AICORE_CLIENT_ID = os.environ["AICORE_CLIENT_ID"]
AICORE_CLIENT_SECRET = os.environ["AICORE_CLIENT_SECRET"]
AICORE_BASE_URL = os.environ["AICORE_BASE_URL"].rstrip("/") + "/v2"
AICORE_RESOURCE_GROUP = os.environ["AICORE_RESOURCE_GROUP"]
AWS_ACCESS_KEY_ID = os.environ.get("AWS_ACCESS_KEY_ID", "")
AWS_SECRET_ACCESS_KEY = os.environ.get("AWS_SECRET_ACCESS_KEY", "")
AWS_REGION = os.environ.get("AWS_REGION", "")
AWS_S3_BUCKET = os.environ.get("AWS_S3_BUCKET", "")
AWS_S3_ENDPOINT = os.environ.get("AWS_S3_ENDPOINT", "").rstrip("/")
ORCHESTRATION_DEPLOYMENT_URL = os.environ["ORCHESTRATION_DEPLOYMENT_URL"]

print(f"Run ID: {RUN_ID}")

Run ID: 20260311-230621


## 2. Prompt Template & Dataset

In [3]:
PROMPT_SPEC = {
    "template": [
        {"role": "system", "content": "You are a premium customer-care reply coach for an online retail brand. Write one support reply that is empathetic, concrete, brand-safe, and short. Do not invent shipment facts. Then add a one-line internal next-best-action note prefixed with 'NBA:'."},
        {"role": "user", "content": "Customer name: {{?customer_name}}\nCustomer message: {{?customer_message}}\nReturn a polished email-ready reply and one internal next-best-action line."},
    ]
}

# 30 training examples (required minimum: 25)
DATASET = [
    {"fields": {"customer_name": "Lena", "customer_message": "My shoes were supposed to arrive yesterday and the tracking hasn't moved for 3 days. This was a birthday gift."}, "answer": "Hi Lena,\n\nI'm very sorry your order has not arrived yet, especially since it was meant to be a birthday gift. I understand how disappointing that is.\n\nI'm checking the shipment status with the carrier now and will follow up with the next confirmed step as quickly as possible. If the delay is confirmed, we will also review the express shipping charge.\n\nBest regards\nNBA: Check carrier movement, confirm delay, and review express-fee refund eligibility."},
    {"fields": {"customer_name": "Marcus", "customer_message": "I paid for express shipping and still don't have my parcel. I need clarity, not another template reply."}, "answer": "Hi Marcus,\n\nI'm sorry your express shipment has not arrived as expected, and I understand you want a clear update.\n\nI'm reviewing the carrier status now and will come back with the next confirmed step instead of a generic response. If the express service was not fulfilled, we will review reimbursement of the shipping charge.\n\nBest regards\nNBA: Verify carrier status immediately and assess express-shipping refund."},
    {"fields": {"customer_name": "Nadia", "customer_message": "Tracking says delivered, but nothing is here. I checked with neighbors already."}, "answer": "Hi Nadia,\n\nI'm sorry to hear the tracking shows delivered while the parcel is still missing.\n\nSince you have already checked with neighbors, the next step is for us to open a proof-of-delivery investigation with the carrier. We will update you as soon as we have the first confirmed findings.\n\nBest regards\nNBA: Open proof-of-delivery investigation and set follow-up with carrier."},
    {"fields": {"customer_name": "Chris", "customer_message": "I ordered running shoes for a race this weekend. The delivery is late, and the last tracking update was two days ago. I'm really disappointed because I paid extra for express shipping."}, "answer": "Hi Chris,\n\nI'm sorry your order is delayed, especially with your race coming up and after you paid for express shipping.\n\nI'm checking the shipment with the carrier now and will update you with the next confirmed step as quickly as possible. If the delay is confirmed, we will review the express shipping charge and the best recovery option.\n\nBest regards\nNBA: Prioritize carrier check and review urgent recovery plus express-fee refund."},
    {"fields": {"customer_name": "Sophie", "customer_message": "I've been waiting for my jacket for over a week now. The tracking page just says 'in transit' with no updates. Can someone please help?"}, "answer": "Hi Sophie,\n\nI'm sorry your jacket has been delayed and the tracking information has not been helpful.\n\nI'm contacting the carrier directly to get a concrete status update for you. I will follow up as soon as I have confirmed information about your shipment's location.\n\nBest regards\nNBA: Contact carrier for shipment location and provide customer with concrete timeline."},
    {"fields": {"customer_name": "James", "customer_message": "My order was supposed to arrive three days ago. I took time off work to receive it and now I don't know when to expect it."}, "answer": "Hi James,\n\nI'm sorry you took time off to receive your order and it still has not arrived. That is understandably frustrating.\n\nI'm checking with the carrier now to get a confirmed delivery window. Once I have that information, I will share it with you so you can plan accordingly.\n\nBest regards\nNBA: Get confirmed delivery window from carrier and communicate to customer."},
    {"fields": {"customer_name": "Elena", "customer_message": "This is the second time my package has been delayed this month. I'm starting to lose trust in your delivery service."}, "answer": "Hi Elena,\n\nI'm sorry to hear this is the second delay you have experienced. I completely understand your frustration and concern about reliability.\n\nI'm looking into your current shipment right now and will also review your previous order to identify what went wrong. We value your trust and want to make this right.\n\nBest regards\nNBA: Investigate current delay, review history, and consider goodwill gesture for repeat issues."},
    {"fields": {"customer_name": "David", "customer_message": "I ordered a gift for my wife's anniversary and it hasn't arrived. The anniversary is tomorrow. Please help!"}, "answer": "Hi David,\n\nI'm so sorry your anniversary gift has not arrived in time. I understand how important this occasion is.\n\nI'm urgently checking with the carrier to see if delivery is possible by tomorrow. If not, I will explore the best alternative solution for you right away.\n\nBest regards\nNBA: Escalate to urgent carrier check and prepare alternative recovery options."},
    {"fields": {"customer_name": "Priya", "customer_message": "The tracking number you gave me doesn't work on the carrier's website. How am I supposed to know where my package is?"}, "answer": "Hi Priya,\n\nI'm sorry the tracking number is not working for you. That should not happen.\n\nI'm verifying the tracking information now and will either provide you with a corrected number or get the status directly from the carrier. I will update you shortly.\n\nBest regards\nNBA: Verify tracking number accuracy and provide corrected information or direct status."},
    {"fields": {"customer_name": "Michael", "customer_message": "My package has been sitting at the local depot for 4 days according to tracking. Why hasn't it been delivered yet?"}, "answer": "Hi Michael,\n\nI'm sorry your package appears stuck at the local depot. That is unusual and frustrating.\n\nI'm contacting the depot directly to find out why delivery has not been attempted and to push for immediate resolution. I will update you with what I learn.\n\nBest regards\nNBA: Contact local depot to resolve hold and arrange immediate delivery."},
    {"fields": {"customer_name": "Anna", "customer_message": "I received a notification that delivery was attempted but I was home all day. No one rang the doorbell."}, "answer": "Hi Anna,\n\nI'm sorry to hear about this. It is frustrating to wait at home and miss a delivery that was never actually attempted.\n\nI'm reporting this issue to the carrier and arranging a redelivery. I will also make sure they have your correct contact details for the next attempt.\n\nBest regards\nNBA: Report false delivery attempt to carrier and schedule confirmed redelivery."},
    {"fields": {"customer_name": "Robert", "customer_message": "My order is a week late and no one has updated me. I had to reach out myself to find out what's going on."}, "answer": "Hi Robert,\n\nI'm sorry you had to reach out yourself after a week without updates. That is not the experience you should have.\n\nI'm investigating the delay now and will provide you with a clear status. I will also ensure you receive proactive updates going forward.\n\nBest regards\nNBA: Investigate delay cause, provide status update, and flag for proactive communication."},
    {"fields": {"customer_name": "Jessica", "customer_message": "The carrier left my package outside in the rain and now the box is damaged. I'm worried about the contents."}, "answer": "Hi Jessica,\n\nI'm sorry to hear your package was left outside in the rain. That should not have happened.\n\nPlease check the contents when you can. If anything is damaged, let me know right away and I will arrange a replacement or refund. I will also report this to the carrier.\n\nBest regards\nNBA: Report delivery issue to carrier and prepare replacement or refund if contents damaged."},
    {"fields": {"customer_name": "Thomas", "customer_message": "I specifically requested the package be left at the back door but it was left at the front where anyone could take it."}, "answer": "Hi Thomas,\n\nI'm sorry your delivery instructions were not followed. I understand the concern about leaving packages in visible locations.\n\nI'm noting this issue with the carrier to ensure your preference is respected in the future. Please let me know if the package is safe or if there are any issues.\n\nBest regards\nNBA: Report delivery instruction failure to carrier and confirm package status with customer."},
    {"fields": {"customer_name": "Maria", "customer_message": "I've called customer service twice about my delayed order and keep getting different answers. Can someone please give me accurate information?"}, "answer": "Hi Maria,\n\nI'm sorry you have received inconsistent information. That must be very frustrating on top of the delay itself.\n\nI'm looking into your order now to get you one clear, accurate answer. I will make sure this information is documented so any future contacts are consistent.\n\nBest regards\nNBA: Investigate thoroughly, provide single accurate status, and document for consistency."},
    {"fields": {"customer_name": "Kevin", "customer_message": "I ordered two items together but only received one. The tracking says everything was delivered."}, "answer": "Hi Kevin,\n\nI'm sorry you only received one of your two items. That is confusing when tracking shows everything delivered.\n\nI'm checking whether the items were shipped together or separately. If the second item is truly missing, I will arrange a replacement right away.\n\nBest regards\nNBA: Check if items shipped separately; if not, arrange replacement for missing item."},
    {"fields": {"customer_name": "Rachel", "customer_message": "My package was returned to sender without any delivery attempt. My address is correct so I don't understand what happened."}, "answer": "Hi Rachel,\n\nI'm sorry your package was returned without a delivery attempt. That should not happen when the address is correct.\n\nI'm investigating why this occurred and will arrange for your order to be reshipped. I will also ensure this does not happen again.\n\nBest regards\nNBA: Investigate return reason, reship order, and verify address is correctly formatted."},
    {"fields": {"customer_name": "Ahmed", "customer_message": "I upgraded to express shipping specifically because I needed the item quickly. It's been five days and still nothing."}, "answer": "Hi Ahmed,\n\nI'm sorry your express shipment has taken five days. That completely defeats the purpose of paying for faster delivery.\n\nI'm checking with the carrier immediately and will refund your express shipping fee regardless of the outcome. I will also update you on the expected delivery.\n\nBest regards\nNBA: Process express shipping refund and escalate carrier check for delivery timeline."},
    {"fields": {"customer_name": "Lisa", "customer_message": "The delivery driver marked my package as delivered to a neighbor but none of my neighbors received anything."}, "answer": "Hi Lisa,\n\nI'm sorry for this confusion. It is frustrating when a package is marked as delivered to a neighbor but cannot be found.\n\nI'm opening an investigation with the carrier to determine where the package was left. If we cannot locate it, I will arrange a replacement for you.\n\nBest regards\nNBA: Open carrier investigation for misdelivery and prepare replacement if unresolved."},
    {"fields": {"customer_name": "Daniel", "customer_message": "I need my order for a work presentation tomorrow. Can you guarantee it will arrive in time?"}, "answer": "Hi Daniel,\n\nI understand you need this for a work presentation tomorrow, which makes timing critical.\n\nI'm checking the current shipment status right now. I will be honest with you about whether tomorrow delivery is realistic, and if not, we will discuss alternative solutions.\n\nBest regards\nNBA: Verify delivery possibility for tomorrow and prepare alternatives if not feasible."},
    {"fields": {"customer_name": "Sarah", "customer_message": "My package has been 'out for delivery' for three days now according to tracking. This doesn't make any sense."}, "answer": "Hi Sarah,\n\nI agree, three days of 'out for delivery' status does not make sense. Something is clearly wrong with either the tracking or the delivery process.\n\nI'm contacting the carrier to get an accurate status and push for actual delivery. I will update you as soon as I have real information.\n\nBest regards\nNBA: Escalate to carrier for accurate status and immediate delivery resolution."},
    {"fields": {"customer_name": "William", "customer_message": "I've been a loyal customer for years and this is the worst delivery experience I've ever had with your company."}, "answer": "Hi William,\n\nI'm truly sorry to hear this has been your worst delivery experience with us, especially as a long-time customer. That is not what you deserve.\n\nI'm personally looking into your order to resolve this as quickly as possible. I will also review what went wrong so we can prevent this in the future.\n\nBest regards\nNBA: Prioritize resolution, investigate root cause, and consider loyalty appreciation gesture."},
    {"fields": {"customer_name": "Emma", "customer_message": "I changed my delivery address a week ago but the package was still sent to my old address. Now someone else has my order."}, "answer": "Hi Emma,\n\nI'm very sorry your address change was not processed correctly and your package went to your old address. I understand how concerning that is.\n\nI'm looking into what happened with the address update. Since someone else received your order, I will arrange a replacement to be sent to your correct address immediately.\n\nBest regards\nNBA: Investigate address update failure, ship replacement to correct address, and verify records."},
    {"fields": {"customer_name": "Oliver", "customer_message": "The estimated delivery date keeps changing every day. First it was Monday, then Wednesday, now it says Friday. What is going on?"}, "answer": "Hi Oliver,\n\nI'm sorry the delivery estimate keeps changing. That inconsistency makes it impossible to plan, and I understand the frustration.\n\nI'm contacting the carrier to get a firm delivery date rather than an estimate. I will share confirmed information with you as soon as I have it.\n\nBest regards\nNBA: Get firm delivery commitment from carrier and communicate confirmed date to customer."},
    {"fields": {"customer_name": "Jennifer", "customer_message": "I'm moving in two days and my package hasn't arrived yet. If it comes after I move, I won't be able to get it."}, "answer": "Hi Jennifer,\n\nI understand the urgency with your move coming up in two days. I want to make sure you receive your package before then.\n\nI'm checking the current shipment status now. If delivery before your move is not possible, I will work with you on redirecting the package to your new address.\n\nBest regards\nNBA: Check delivery timeline and arrange redirect to new address if needed."},
    {"fields": {"customer_name": "Brian", "customer_message": "Your website said delivery in 2-3 days. It's been 8 days. This is false advertising."}, "answer": "Hi Brian,\n\nI'm sorry the delivery has taken far longer than the 2-3 days promised on our website. That expectation was clearly not met and I understand your frustration.\n\nI'm investigating why this shipment was delayed and will provide you with a status update. I will also review compensation options for the inconvenience.\n\nBest regards\nNBA: Investigate delay cause, provide status, and offer appropriate compensation."},
    {"fields": {"customer_name": "Michelle", "customer_message": "I ordered medicine that I need urgently and the delivery is late. This is affecting my health."}, "answer": "Hi Michelle,\n\nI'm very sorry your order is delayed, especially given that you need this medicine urgently. Your health is a priority and I take this seriously.\n\nI'm escalating this immediately to get you a status update as fast as possible. If the delivery cannot happen today, please let me know so we can discuss alternatives.\n\nBest regards\nNBA: Escalate as urgent medical need, get immediate status, explore expedited solutions."},
    {"fields": {"customer_name": "Steven", "customer_message": "I was promised a refund for my delayed order but it's been two weeks and I haven't seen anything."}, "answer": "Hi Steven,\n\nI'm sorry the refund you were promised has not arrived after two weeks. That is an unacceptable delay on top of the original delivery issue.\n\nI'm looking into the refund status right now and will make sure it is processed. I will update you with confirmation once it is done.\n\nBest regards\nNBA: Verify refund status, process if pending, and confirm completion to customer."},
    {"fields": {"customer_name": "Nicole", "customer_message": "The delivery window was 9am-12pm. It's now 4pm and still no package. I wasted my whole day waiting."}, "answer": "Hi Nicole,\n\nI'm sorry you waited all day within the promised delivery window and still have not received your package. That is completely unacceptable.\n\nI'm contacting the carrier to find out what happened and when you can actually expect delivery. I will also address the wasted time with appropriate compensation.\n\nBest regards\nNBA: Get immediate carrier update, arrange same-day delivery if possible, offer compensation."},
]

# Save datasets
SHARED_DATASET_FILENAME = "cx_delivery_delay_dataset.json"
EVAL_DATASET_FILENAME = "cx_delivery_delay_eval_dataset.json"
OBJECT_STORE_PATH_PREFIX = f"hackathon/prompt-optimization/{RUN_ID}"

local_dataset_path = LOCAL_WORKDIR / SHARED_DATASET_FILENAME
local_dataset_path.write_text(json.dumps(DATASET, indent=2), encoding="utf-8")

eval_dataset = [{**item["fields"], "expected_output": item["answer"]} for item in DATASET]
local_eval_path = LOCAL_WORKDIR / EVAL_DATASET_FILENAME
local_eval_path.write_text(json.dumps(eval_dataset, indent=2), encoding="utf-8")

print(f"Dataset: {len(DATASET)} examples saved")

Dataset: 29 examples saved


## 3. Live Orchestration Test

In [4]:
template = Template(template=[SystemMessage(content=PROMPT_SPEC["template"][0]["content"]), UserMessage(content=PROMPT_SPEC["template"][1]["content"])])
llm = LLMModelDetails(name="gpt-4o-mini", params={"temperature": 0.2, "max_completion_tokens": 220})
service = OrchestrationService(config=OrchestrationConfig(modules=ModuleConfig(prompt_templating=PromptTemplatingModuleConfig(prompt=template, model=llm))))

result = service.run(placeholder_values={"customer_name": "Chris", "customer_message": "I ordered running shoes for a race this weekend. The delivery is late and I paid extra for express shipping."})
print("Generated Reply:\n" + result.final_result.choices[0].message.content)

Generated Reply:
Subject: We're Here to Help with Your Order

Dear Chris,

I completely understand how important it is to receive your running shoes on time, especially with your race this weekend. I’m truly sorry for the delay in your delivery, especially after you opted for express shipping. 

Please rest assured that we are looking into this matter and will do our best to resolve it as quickly as possible. If you have any further questions or need assistance, feel free to reach out.

Thank you for your patience and understanding.

Best regards,  
[Your Name]  
Customer Care Team

NBA: Check the order status and escalate to shipping department for priority resolution.


## 4. AI Core Client

In [5]:
class AICore:
    def __init__(self):
        self._token, self._expiry = None, 0
        self._http = httpx.Client(timeout=120, follow_redirects=True)
    
    def close(self): self._http.close()
    
    def _get_token(self):
        if self._token and time.time() < self._expiry - 60: return self._token
        r = self._http.post(f"{AICORE_AUTH_URL.rstrip('/')}/oauth/token", data={"grant_type": "client_credentials"}, auth=(AICORE_CLIENT_ID, AICORE_CLIENT_SECRET))
        r.raise_for_status()
        self._token, self._expiry = r.json()["access_token"], time.time() + r.json().get("expires_in", 1800)
        return self._token
    
    def req(self, method, path, body=None, params=None, rg=True):
        headers = {"Authorization": f"Bearer {self._get_token()}", "Content-Type": "application/json"}
        if rg: headers["AI-Resource-Group"] = AICORE_RESOURCE_GROUP
        r = self._http.request(method, f"{AICORE_BASE_URL}{path}", json=body, params=params, headers=headers)
        if r.status_code >= 400: raise RuntimeError(f"{method} {path}: {r.status_code}\n{r.text}")
        return r.json() if r.content and "json" in r.headers.get("Content-Type", "") else r.text if r.content else None
    
    def poll(self, getter, obj_id, terminals, timeout=900):
        start = time.time()
        while time.time() - start < timeout:
            p = getter(obj_id)
            status = p.get("status", "UNKNOWN").upper()
            print(f"  {obj_id}: {status}")
            if status in {s.upper() for s in terminals}: return p
            time.sleep(5)
        raise TimeoutError(f"Timeout: {obj_id}")

client = AICore()
print("Client initialized")

Client initialized


## 5. Setup Resources & Upload Data

In [6]:
# Ensure resource group
try: client.req("POST", "/admin/resourceGroups", {"resourceGroupId": AICORE_RESOURCE_GROUP}, rg=False); print(f"Created: {AICORE_RESOURCE_GROUP}")
except: print(f"Exists: {AICORE_RESOURCE_GROUP}")

# Upload datasets to S3
s3 = boto3.client('s3', aws_access_key_id=AWS_ACCESS_KEY_ID, aws_secret_access_key=AWS_SECRET_ACCESS_KEY, region_name=AWS_REGION)
for lp, fn in [(local_dataset_path, SHARED_DATASET_FILENAME), (local_eval_path, EVAL_DATASET_FILENAME)]:
    key = f"{OBJECT_STORE_PATH_PREFIX}/datasets/{fn}"
    s3.put_object(Bucket=AWS_S3_BUCKET, Key=key, Body=lp.read_bytes(), ContentType="application/json")
    print(f"Uploaded: {fn}")

ARTIFACT_ROOT = f"ai://default/{OBJECT_STORE_PATH_PREFIX}/datasets"
print(f"Artifact root: {ARTIFACT_ROOT}")

Exists: default
Uploaded: cx_delivery_delay_dataset.json
Uploaded: cx_delivery_delay_eval_dataset.json
Artifact root: ai://default/hackathon/prompt-optimization/20260311-230621/datasets


## 6. Register Prompt Templates

In [7]:
PROMPT_NAME = f"{USE_CASE_NAME}-base-{RUN_ID}"

# For optimizer
opt_prompt = client.req("POST", "/lm/promptTemplates", {"name": PROMPT_NAME, "version": "0.0.1", "scenario": "genai-optimizations", "spec": PROMPT_SPEC})
print(f"Optimizer prompt: genai-optimizations/{PROMPT_NAME}:0.0.1")

# For evaluation
eval_prompt = client.req("POST", "/lm/promptTemplates", {"name": f"{PROMPT_NAME}-eval", "version": "0.0.1", "scenario": "genai-evaluations", "spec": PROMPT_SPEC})
eval_prompt_id = eval_prompt.get("id")
print(f"Eval prompt ID: {eval_prompt_id}")

Optimizer prompt: genai-optimizations/delivery-delay-reply-coach-base-20260311-230621:0.0.1
Eval prompt ID: 7ae5040e-8d7f-4d5e-9d36-06b430675efb


## 7. Run Evaluation

In [8]:
# # Register artifact
# eval_artifact = client.req("POST", "/lm/artifacts", {
#     "name": f"cx-delay-dataset-{RUN_ID}", "kind": "other", "scenarioId": "genai-evaluations",
#     "url": ARTIFACT_ROOT, "labels": [{"key": "ext.ai.sap.com/prompt-evaluation", "value": "true"}],
#     "description": "Eval dataset"
# })
# eval_artifact_id = eval_artifact["id"]

# # Create config
# eval_config = client.req("POST", "/lm/configurations", {
#     "name": f"{USE_CASE_NAME}-eval-{RUN_ID}", "scenarioId": "genai-evaluations", "executableId": "genai-evaluations-simplified",
#     "inputArtifactBindings": [{"key": "datasetFolder", "artifactId": eval_artifact_id}],
#     "parameterBindings": [
#         {"key": "repetitions", "value": "1"}, {"key": "orchestrationDeploymentURL", "value": ORCHESTRATION_DEPLOYMENT_URL},
#         {"key": "metrics", "value": "Pointwise Answer Relevance,Pointwise Conciseness"},
#         {"key": "testDataset", "value": json.dumps({"path": EVAL_DATASET_FILENAME, "type": "json"})},
#         {"key": "promptTemplate", "value": eval_prompt_id}, {"key": "models", "value": "gpt-4o:2024-08-06"},
#         {"key": "orchestrationRegistryIds", "value": ""}
#     ]
# })

# # Execute and poll
# eval_exec = client.req("POST", "/lm/executions", {"configurationId": eval_config["id"]})
# print(f"Evaluation started: {eval_exec['id']}")
# client.poll(lambda x: client.req("GET", f"/lm/executions/{x}"), eval_exec["id"], {"COMPLETED", "FAILED", "DEAD"})

# # Get metrics
# try:
#     metrics = client.req("GET", "/lm/metrics", params={"tagFilters": f"evaluation.ai.sap.com/child-of={eval_exec['id']}"})
#     print(f"Metrics: {json.dumps(metrics, indent=2)}")
# except Exception as e: print(f"Metrics error: {e}")

## 8. Run Optimization

In [9]:
# Register artifact
opt_artifact = client.req("POST", "/lm/artifacts", {
    "name": f"{USE_CASE_NAME}-opt-dataset-{RUN_ID}", "kind": "dataset", "scenarioId": "genai-optimizations",
    "url": ARTIFACT_ROOT, "description": "Optimizer dataset"
})

# Create config
# Note: targetModels and targetPromptMapping must use consistent model naming
TARGET_MODEL = "gpt-4o:2024-08-06"
TARGET_MODEL_MAPPED = "openai/gpt-4o-2024-08-06"  # Format used in targetPromptMapping

opt_config = client.req("POST", "/lm/configurations", {
    "name": f"{USE_CASE_NAME}-opt-{RUN_ID}", "scenarioId": "genai-optimizations", "executableId": "genai-optimizations",
    "inputArtifactBindings": [{"key": "prompt-data", "artifactId": opt_artifact["id"]}],
    "parameterBindings": [
        {"key": "dataset", "value": SHARED_DATASET_FILENAME}, 
        {"key": "optimizationMetric", "value": "LLMaaJ:Sem_Sim_1"},
        {"key": "basePrompt", "value": f"genai-optimizations/{PROMPT_NAME}:0.0.1"}, 
        {"key": "baseModel", "value": TARGET_MODEL},
        {"key": "targetModels", "value": TARGET_MODEL},
        {"key": "targetPromptMapping", "value": f"{TARGET_MODEL_MAPPED}={USE_CASE_NAME}-opt-gpt4o-{RUN_ID}:0.0.1"}
    ]
})

# Execute and poll
opt_exec = client.req("POST", "/lm/executions", {"configurationId": opt_config["id"]})
opt_exec_id = opt_exec["id"]
print(f"Optimization started: {opt_exec_id}")
final_status = client.poll(lambda x: client.req("GET", f"/lm/executions/{x}"), opt_exec_id, {"COMPLETED", "FAILED", "DEAD"})
print(f"Final status: {final_status.get('status', 'UNKNOWN')}")

Optimization started: e0cd76079dac77e7
  e0cd76079dac77e7: UNKNOWN
  e0cd76079dac77e7: UNKNOWN
  e0cd76079dac77e7: UNKNOWN
  e0cd76079dac77e7: UNKNOWN
  e0cd76079dac77e7: UNKNOWN
  e0cd76079dac77e7: RUNNING
  e0cd76079dac77e7: RUNNING
  e0cd76079dac77e7: RUNNING
  e0cd76079dac77e7: RUNNING
  e0cd76079dac77e7: RUNNING
  e0cd76079dac77e7: RUNNING
  e0cd76079dac77e7: RUNNING
  e0cd76079dac77e7: RUNNING
  e0cd76079dac77e7: RUNNING
  e0cd76079dac77e7: RUNNING
  e0cd76079dac77e7: RUNNING
  e0cd76079dac77e7: RUNNING
  e0cd76079dac77e7: RUNNING
  e0cd76079dac77e7: RUNNING
  e0cd76079dac77e7: RUNNING
  e0cd76079dac77e7: RUNNING
  e0cd76079dac77e7: RUNNING
  e0cd76079dac77e7: RUNNING
  e0cd76079dac77e7: RUNNING
  e0cd76079dac77e7: RUNNING
  e0cd76079dac77e7: RUNNING
  e0cd76079dac77e7: RUNNING
  e0cd76079dac77e7: RUNNING
  e0cd76079dac77e7: RUNNING
  e0cd76079dac77e7: RUNNING
  e0cd76079dac77e7: RUNNING
  e0cd76079dac77e7: RUNNING
  e0cd76079dac77e7: RUNNING
  e0cd76079dac77e7: RUNNING
  e0cd760

## 9. View Results

In [10]:
# Fetch and display optimization results
def display_optimization_results(results):
    print("=" * 60)
    print("OPTIMIZATION RESULTS")
    print("=" * 60)
    print(f"Status: {results.get('job_status', 'N/A')}")
    
    # Origin model info
    origin = results.get("origin_model", {})
    if origin:
        print(f"\n--- Baseline (Origin Model) ---")
        print(f"Model: {origin.get('model_name', 'N/A')}")
        print(f"Score: {origin.get('score', 'N/A')}")
        sys_prompt = origin.get("system_prompt", "")
        if sys_prompt:
            print(f"System Prompt:\n{sys_prompt[:500]}{'...' if len(sys_prompt) > 500 else ''}")
    
    # Target models (optimized prompts)
    for i, target in enumerate(results.get("target_models", []), 1):
        print(f"\n--- Target Model {i} ---")
        print(f"Model: {target.get('model_name', 'N/A')}")
        pre = target.get("pre_optimization_score", 0)
        post = target.get("post_optimization_score", 0)
        improvement = post - pre if isinstance(pre, (int, float)) and isinstance(post, (int, float)) else "N/A"
        print(f"Pre-optimization Score: {pre}")
        print(f"Post-optimization Score: {post}")
        print(f"Improvement: +{improvement:.4f}" if isinstance(improvement, float) else f"Improvement: {improvement}")
        
        sys_prompt = target.get("system_prompt", "")
        if sys_prompt:
            print(f"\nOptimized System Prompt:\n{sys_prompt}")

try:
    results_path = f"default/{opt_exec_id}/result-data/results.json"
    raw_results = client.req("GET", f"/lm/dataset/files/{quote(results_path, safe='')}")
    
    # Parse JSON if returned as string
    results = json.loads(raw_results) if isinstance(raw_results, str) else raw_results
    display_optimization_results(results)
    
except Exception as e:
    print(f"Could not fetch results: {e}")
    print("Note: Results may not be available yet if the optimization is still running.")

client.close()
print("\nDone!")

OPTIMIZATION RESULTS
Status: completed

--- Baseline (Origin Model) ---
Model: gpt-4o:2024-08-06
Score: 0.586
System Prompt:
You are a premium customer-care reply coach for an online retail brand. Write one support reply that is empathetic, concrete, brand-safe, and short. Do not invent shipment facts. Then add a one-line internal next-best-action note prefixed with 'NBA:'.

--- Target Model 1 ---
Model: gpt-4o:2024-08-06
Pre-optimization Score: 0.517
Post-optimization Score: 0.793
Improvement: +0.2760

Optimized System Prompt:
You are an expert customer-care specialist for a premium online retail brand. Your objective is to resolve customer issues by producing one empathetic, brand-safe customer-facing reply and one internal next-best-action (NBA) note per request.

Principles:
- Verify first: Before offering any remedy, confirm the shipment, order, or refund status via reliable systems or carrier contact.
- Remedy gating: Only propose or initiate remedies (refunds, replacements, comp